# Lab 1 — Understand the curve number

**Twenty minutes.** This lab establishes the theoretical and numerical
framework used by both spatial-data pathways.

The guided analysis is organized into eight explicit steps:

1. define the event water-balance model;
2. translate curve number to retention and initial abstraction;
3. evaluate the piecewise runoff equation;
4. compare distributed and lumped watershed representations;
5. examine how the compositing difference varies with storm depth;
6. treat lambda and curve number as a paired calibration;
7. invert observed rainfall and runoff to event curve numbers; and
8. fit and interpret an asymptotic curve number.

Each step states the theoretical purpose, shows the intermediate
quantities, explains the corresponding `cnkit` operation, and ends with
the interpretation expected from the participant. During the
twenty-minute laboratory, prioritize the code cells and retain the
surrounding material as a technical reference.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cnkit import (
    CN_from_PQ,
    S_from_CN,
    cn05_from_cn20,
    composite_runoff,
    fit_asymptotic,
    runoff,
)


## Step 1 — Define the event water-balance model

The curve-number method is an event-scale, lumped rainfall–runoff
relation. For rainfall depth $P$, direct-runoff depth $Q$, potential
maximum retention $S$, and initial abstraction $I_a$, the standard
equations in inch units are

\[
S = \frac{1000}{CN} - 10,
\qquad I_a = \lambda S,
\]

\[
Q = \begin{cases}
0, & P \le I_a,\\[4pt]
\dfrac{(P-I_a)^2}{P+(1-\lambda)S}, & P>I_a.
\end{cases}
\]

A curve number is therefore a dimensionless transformation of $S$;
it is not a directly observed land-surface property. Larger CN implies
smaller retention, a smaller rainfall threshold, and greater runoff for
the same event. The conventional value \(\lambda=0.20\) specifies the
assumed fraction of retention that must be satisfied before runoff
begins.


## Step 2 — Translate CN into retention and initial abstraction

**Why this step is needed.** CN is used by the runoff equation only
after it has been transformed to retention. Displaying retention and
the abstraction threshold makes the physical consequence of a CN
choice visible before runoff is calculated.

**How the library performs it.** `S_from_CN` converts its input to a
numeric array, checks that every curve number is in the interval
$0<CN\leq100$, and applies $1000/CN-10$ element by element. It does
not assign lambda, so the notebook calculates $I_a=\lambda S$
explicitly.

**What to inspect.** Compare the change in retention with the change in
CN. The transformation is nonlinear: a ten-unit CN change does not
represent a constant change in storage across the CN range.


In [ ]:
curve_numbers = np.array([55, 70, 85, 98], dtype=float)
retention = S_from_CN(curve_numbers)
theory_table = pd.DataFrame(
    {
        "curve_number": curve_numbers,
        "retention_S_in": retention,
        "initial_abstraction_Ia_in": 0.20 * retention,
    }
)
theory_table.round(3)


**Interpretation.** The threshold is mathematically consequential. At CN 70 and
$\lambda=0.20$, $I_a$ is approximately 0.86 inches; an event below
that depth produces zero direct runoff in the model. This is a model
statement, not a claim that no water moves within the watershed.


## Step 3 — Evaluate and verify the runoff equation

**Why this step is needed.** Reproducing a library result from the
published equation confirms the units, threshold convention, and
numerical interpretation before the method is applied spatially.

**How the library performs it.** `cnkit.runoff` validates rainfall, CN,
and lambda; broadcasts compatible scalar or array inputs; calculates
$S$ and $I_a$; and uses a piecewise mask to return zero where
$P\leq I_a$. For the remaining elements it evaluates the rational
runoff equation. The library does not select CN, lambda, rainfall, or
the spatial unit of analysis.


In [ ]:
def runoff_written_out(P, cn, lam=0.20):
    P = np.asarray(P, dtype=float)
    S = 1000.0 / float(cn) - 10.0
    Ia = lam * S
    return np.where(P > Ia, (P - Ia) ** 2 / (P + (1.0 - lam) * S), 0.0)

storms_check = np.array([0.50, 1.00, 2.00, 4.00])
by_equation = runoff_written_out(storms_check, 70, lam=0.20)
by_library = runoff(storms_check, 70, lam=0.20)
verification = pd.DataFrame(
    {
        "P_in": storms_check,
        "Q_equation_in": by_equation,
        "Q_cnkit_in": by_library,
        "absolute_difference": np.abs(by_equation - by_library),
    }
)
assert np.allclose(by_equation, by_library)
verification.round(6)


**Interpretation.** The assertion checks every storm depth numerically.
Values below the threshold should be exactly zero, and the remaining
values should agree to floating-point precision. This establishes that
later differences arise from parameter or spatial choices rather than a
second equation.


## Step 4 — Compare distributed and lumped watershed representations

The runoff equation is nonlinear in CN because CN is first transformed
to $S$, enters both the threshold and denominator, and appears inside
a squared numerator. Consequently,

\[
Q\!\left(P,\sum_i w_i CN_i\right)
\ne \sum_i w_i Q(P,CN_i)
\]

in general. The left side is a lumped calculation; the right side
computes runoff for each hydrologically distinct subarea and then
aggregates runoff volume. Both are reproducible calculations, but they
represent different spatial models.

Sixty percent connected impervious cover at CN 98 is combined with
forty percent woods at CN 55 under a one-inch storm. `composite_runoff`
returns three results in a fixed order:

1. runoff by subarea, subsequently area weighted;
2. CN area weighted first, followed by one runoff calculation; and
3. retention $S$ area weighted first, converted back to CN, followed
   by one runoff calculation.

**How the library performs it.** The function validates that CN and
area arrays are finite, positive, and equal in length; normalizes area
to weights; calls `runoff` on each subarea; and calculates the weighted
sum of runoff. It then calls `composite_cn` and
`composite_cn_via_S` to produce the two lumped comparisons. The return
order is distributed runoff, runoff from weighted CN, and runoff from
weighted retention.


In [ ]:
P_weighting = 1.0
subarea_cn = np.array([98.0, 55.0])
subarea_fraction = np.array([0.60, 0.40])

q_distributed, q_weighted_cn, q_weighted_s = composite_runoff(
    P_weighting, subarea_cn, subarea_fraction
)

subarea_detail = pd.DataFrame(
    {
        "description": ["connected impervious", "woods on HSG B"],
        "area_fraction": subarea_fraction,
        "CN": subarea_cn,
        "S_in": S_from_CN(subarea_cn),
        "Ia_in": 0.20 * S_from_CN(subarea_cn),
        "subarea_Q_in": runoff(P_weighting, subarea_cn),
    }
)
subarea_detail["runoff_contribution_in"] = (
    subarea_detail.area_fraction * subarea_detail.subarea_Q_in
)
subarea_detail.round(4)


In [ ]:
weighting = pd.Series(
    {
        "runoff by subarea, then area-weight": q_distributed,
        "area-weight CN, then compute runoff": q_weighted_cn,
        "area-weight S, then compute runoff": q_weighted_s,
    },
    name="runoff_inches",
)
print(weighting.round(4))
print("distributed / weighted-CN ratio: %.1f" % (q_distributed / q_weighted_cn))


**Interpretation.** The wooded subarea remains below its
initial-abstraction threshold while the impervious subarea is already
producing runoff. A lumped parameter removes that threshold contrast
before the nonlinear equation is evaluated. This explains why the
difference is largest for smaller storms and heterogeneous watersheds.


## Step 5 — Examine how compositing depends on storm depth

**Why this step is needed.** A single design storm shows one point on a
nonlinear response. Repeating the same three spatial representations
across rainfall depths reveals whether their difference is structural
or specific to the selected storm.

**How the library performs it.** The notebook calls
`composite_runoff` once per rainfall depth while keeping CN values,
areas, and lambda fixed. Only $P$ changes. Each plotted line therefore
represents one compositing convention under otherwise identical input.


In [ ]:
storms = np.linspace(0.25, 6.0, 48)
values = np.array(
    [composite_runoff(p, subarea_cn, subarea_fraction) for p in storms]
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(storms, values[:, 0], lw=2.5, label="distributed runoff")
ax.plot(storms, values[:, 1], lw=2, label="weighted CN")
ax.plot(storms, values[:, 2], lw=2, label="weighted S")
ax.set(xlabel="storm depth, inches", ylabel="runoff depth, inches")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


**Interpretation.** Near the abstraction thresholds, some subareas
produce runoff while others do not, so early parameter aggregation has
its greatest effect. With larger storms, every subarea contributes and
the relative separation generally decreases. Record both storm depth
and spatial convention when reporting the comparison.


## Step 6 — Treat lambda and CN as a paired calibration

Lambda is not an independent switch applied after CN has been selected.
Event-derived and table-derived curve numbers are conditional on the
lambda used in the runoff equation. A CN calibrated with
\(\lambda=0.20\) should therefore be converted or refitted before it is
used with \(\lambda=0.05\).

`cn05_from_cn20` implements the Hawkins et al. (2003) empirical
conversion. The converted CN is numerically lower because the smaller
initial-abstraction ratio permits runoff to begin earlier. Similar
runoff response—not equal CN—is the comparison to make.

Difficult Run has a 2019 table composite near 75.5 and a fitted
asymptotic value of 69.7. The lambda conversion below is a third number:
it describes the same response convention under lambda 0.05 rather than
0.20.

**How the library performs it.** `cn05_from_cn20` applies the published
empirical conversion to each validated CN value. This is a parameter
conversion, not a second runoff calculation. The notebook then calls
`runoff` with each CN–lambda pair at the same storm depth so that the
resulting response can be compared on a common basis.


In [ ]:
P = 3.0
table_cn20 = 75.5
fitted_cn20 = 69.7
table_cn05 = float(cn05_from_cn20(table_cn20))

comparison = pd.DataFrame(
    [
        ["table", 0.20, table_cn20, float(runoff(P, table_cn20, lam=0.20))],
        ["fitted to gage", 0.20, fitted_cn20, float(runoff(P, fitted_cn20, lam=0.20))],
        ["same table response, converted", 0.05, table_cn05, float(runoff(P, table_cn05, lam=0.05))],
    ],
    columns=["basis", "lambda", "curve_number", "runoff_in"],
)
comparison.round(4)


**Interpretation.** The three rows have different evidentiary bases: a
spatial lookup, a rainfall–runoff fit, and a conversion between
equation conventions. They should be labelled accordingly rather than
described as competing measurements of one fixed property.


## Step 7 — Invert observed rainfall and runoff to event curve numbers

For an observed event pair \((P,Q)\), `CN_from_PQ` algebraically inverts
the same runoff equation for a specified lambda. Each valid event
produces an event-derived CN. These values vary with storm depth,
antecedent state, measurement error, and model adequacy; the variation
is information rather than a reason to average immediately.

**How the library performs it.** `CN_from_PQ` broadcasts rainfall and
runoff arrays, identifies physically admissible events, solves the
quadratic relation for retention, and transforms retention to CN. An
event receives `NaN` when the inputs do not support a physical inverse,
such as non-positive runoff or runoff exceeding rainfall. Lambda is an
explicit argument because it changes the inverse solution.


In [ ]:
events = pd.read_csv(DATA_DIR / "events_01646000.csv")
events["event_CN"] = CN_from_PQ(
    events.P_in.values, events.Q_in.values, lam=0.20
)
events["valid_inverse"] = np.isfinite(events.event_CN)

print("event records:       ", len(events))
print("valid inverse events:", int(events.valid_inverse.sum()))
events[
    ["P_in", "Q_in", "runoff_ratio", "event_CN", "valid_inverse"]
].head(12).round(3)


**Interpretation.** Event CN is derived from measured quantities under
a specified model convention. The event table should therefore retain
rainfall, runoff, runoff ratio, lambda, and validity status beside the
transformed value.


## Step 8 — Fit and interpret an asymptotic curve number

`fit_asymptotic` then fits the Hawkins standard relation

\[
CN(P)=CN_{\infty}+(100-CN_{\infty})e^{-kP}
\]

by nonlinear least squares. \(CN_{\infty}\) is the large-storm
asymptote, $k$ controls the rate of approach, and $R^2$ describes
how much of the event-CN variation is explained by this specific
functional form. The fit does not establish that the watershed has a
unique physical CN.

**How the library performs it.** The function calls `CN_from_PQ`,
retains finite event values, checks that the requested minimum event
count is available, and uses bounded nonlinear least squares to
estimate $CN_{\infty}$ and $k$. It returns a structured result with
the model name, coefficients, event count, RMSE, and $R^2$. Its
`predict` method evaluates the selected fitted model at new rainfall
depths.


In [ ]:
fit = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.20)

valid = events.valid_inverse.values
p_line = np.linspace(events.P_in.min(), events.P_in.max(), 200)
fitted_line = fit.predict(p_line)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(events.P_in.values[valid], events.event_CN.values[valid], s=12, alpha=0.25,
           label="event-derived CN")
ax.plot(p_line, fitted_line, color="#b24d35", lw=2.5,
        label="Hawkins standard fit")
ax.axhline(table_cn20, color="#17274f", ls="--", lw=1.8,
           label="2019 table composite")
ax.set(xlabel="event rainfall P, inches", ylabel="event-derived curve number")
ax.set_ylim(0, 102)
ax.grid(alpha=0.2)
ax.legend()
plt.show()

print("event records:", len(events))
print("events fitted:", fit.n_events)
print("CN infinity:   %.3f" % fit.cn_inf)
print("decay k:       %.3f" % fit.k)
print("R squared:     %.3f" % fit.r2)


**Interpretation.** Compare the event cloud, fitted curve, table value,
fitted event count, and diagnostics together. A numerical asymptote is
meaningful only to the extent that the selected response form is
supported over the observed rainfall range.


## Method audit — Library operation and analyst decision

| Call | Library operation | Analyst responsibility |
|---|---|---|
| `S_from_CN` | Applies the CN-to-retention transformation | Establish the basis and scale of CN |
| `runoff` | Applies the piecewise event equation | Select $P$, CN, lambda, and spatial representation |
| `composite_runoff` | Returns distributed, weighted-CN, and weighted-S calculations | Choose and justify the compositing convention |
| `cn05_from_cn20` | Applies the published empirical parameter conversion | Keep the converted CN paired with lambda 0.05 |
| `CN_from_PQ` | Inverts the event equation | Verify rainfall, direct-runoff separation, and event selection |
| `fit_asymptotic` | Fits a named CN-versus-rainfall model | Evaluate fit adequacy and interpretability |

The library makes the transformations reproducible; it does not make
the scientific choices interchangeable.


## Report-out

Bring back:

1. The three runoff depths from the weighting example.
2. Which convention you would report for a heterogeneous watershed.
3. One sentence explaining why lambda must be reported with CN.
4. One sentence distinguishing the table composite from
   \(CN_{\infty}\).

**Source anchors:** NEH-630 Chapter 10 equations 10-1 and 10-11;
TR-55 Worksheet 2; Woodward et al. (2003), DOI
`10.1061/40685(2003)308`; Hawkins et al. (2003), DOI
`10.1061/(ASCE)1084-0699(2003)8:6(445)`.
